# 3D FDTD Postprocessing — Fishbone Grating Group Index

Loads pre-computed Meep 3D FDTD results from `.npz` files and extracts group index $n_g$ via three independent methods:

| Method | Description |
|--------|-------------|
| **1. Broadband peak delay** | Hilbert-envelope peak arrival time at both monitors; $n_g = \Delta t / L$ |
| **2. Bandpass filter** | Narrow Gaussian filter at each $\lambda$, envelope peak delay; $n_g(\lambda)$ |
| **3. Spectral phase** | Transfer function $H(\omega)$ unwrapped phase derivative; $n_g = -\frac{1}{L}\frac{d\phi}{d\omega}$ |

**Datasets:**
- `fdtd_71nm_data_top1.npz` — $a = 496.0$ nm, window 1512–1531 nm
- `fdtd_71nm_data_top2.npz` — $a = 505.4$ nm, window 1542–1558 nm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.animation as animation
from scipy.signal import hilbert
from IPython.display import HTML
import os

# ── Load both NPZ files ───────────────────────────────────────────────
NPZ_FILES = [
    'fdtd_71nm_data_top1.npz',
    #'fdtd_71nm_data_top2.npz',
]

datasets = []
for fname in NPZ_FILES:
    d = np.load(fname, allow_pickle=True)
    ds = {k: d[k] for k in d.files}
    # Scalar fields to plain Python for convenience
    for key in ['a_nm', 'L_mon', 'mon1_x', 'mon2_x', 'src_x',
                'grating_start', 'grating_end',
                'snap_region_x', 'snap_region_y',
                'sx', 'sy', 'sz',
                'LAMBDA_MIN_NM', 'LAMBDA_MAX_NM',
                'NUM_PERIODS', 'RESOLUTION']:
        if key in ds and ds[key].ndim == 0:
            ds[key] = float(ds[key])
    ds['label'] = fname.replace('.npz', '')
    datasets.append(ds)
    print(f"Loaded {fname}:")
    print(f"  a = {ds['a_nm']:.1f} nm, window = {ds['LAMBDA_MIN_NM']:.0f}–{ds['LAMBDA_MAX_NM']:.0f} nm")
    print(f"  eps_xy: {ds['eps_xy'].shape},  eps_xz: {ds['eps_xz'].shape}")
    print(f"  ey_snapshots: {ds['ey_snapshots'].shape}")
    print("No data provided. Please provide the necessary data to generate the animation.")

## 1. Structure Geometry

- **Top view** (`eps_xy`): permittivity at the centre of the Si slab ($z = 0$). Bright = Si, dark = air.
- **Cross-section** (`eps_xz`): permittivity at $y = 0$ showing the slab stack and partial etch.

In [ ]:
N_SI = 3.48
eps_Si = N_SI ** 2

fig, axes = plt.subplots(2, 2, figsize=(18, 7),
                          gridspec_kw={'height_ratios': [2, 1]})

for col, ds in enumerate(datasets):
    a  = ds['a_nm']
    rx = ds['snap_region_x'] * a / 1e3   # µm
    ry = ds['snap_region_y'] * a / 1e3   # µm
    rz = ds['sz'] * a / 1e3              # µm

    ext_xy = [-rx/2, rx/2, -ry/2, ry/2]
    ext_xz = [-rx/2, rx/2, -rz/2, rz/2]

    # --- Top view (eps_xy) ---
    ax = axes[0, col]
    im = ax.imshow(ds['eps_xy'].T, origin='lower', cmap='inferno',
                   extent=ext_xy, aspect='auto',
                   vmin=1, vmax=eps_Si)
    # Monitor positions
    m1 = ds['mon1_x'] * a / 1e3
    m2 = ds['mon2_x'] * a / 1e3
    ax.axvline(m1, color='cyan',  ls='--', lw=1, label=f'Mon1 x={m1:.2f}µm')
    ax.axvline(m2, color='lime',  ls='--', lw=1, label=f'Mon2 x={m2:.2f}µm')
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_title(f'{ds["label"]}\nTop view (eps_xy, z=0)  a={a:.1f} nm')
    ax.legend(fontsize=8, loc='upper right')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='ε')

    # --- Cross-section (eps_xz) ---
    ax = axes[1, col]
    im2 = ax.imshow(ds['eps_xz'].T, origin='lower', cmap='inferno',
                    extent=ext_xz, aspect='auto',
                    vmin=1, vmax=eps_Si)
    ax.axvline(m1, color='cyan',  ls='--', lw=0.8)
    ax.axvline(m2, color='lime',  ls='--', lw=0.8)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('z (µm)')
    ax.set_title(f'Cross-section (eps_xz, y=0)')
    plt.colorbar(im2, ax=ax, fraction=0.03, pad=0.02, label='ε')

plt.suptitle('Fishbone Grating — Dielectric Structure', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fdtd_geometry.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Electric Field Snapshots ($E_y$)

Selected frames from the recorded $E_y$ movie at the $z = 0$ plane. The dashed cyan/lime lines mark the input/output monitor positions.

In [ ]:
N_FRAMES_SHOW = 6   # frames to display per dataset

for ds in datasets:
    a  = ds['a_nm']
    rx = ds['snap_region_x'] * a / 1e3
    ry = ds['snap_region_y'] * a / 1e3
    ext_xy = [-rx/2, rx/2, -ry/2, ry/2]

    ey_stack = ds['ey_snapshots']
    vmax = np.percentile(np.abs(ey_stack), 99.5)
    n_snaps = ey_stack.shape[0]
    idxs = np.linspace(0, n_snaps - 1, N_FRAMES_SHOW, dtype=int)

    fig, axes = plt.subplots(N_FRAMES_SHOW, 1,
                             figsize=(16, 1.8 * N_FRAMES_SHOW),
                             constrained_layout=True)
    for i, idx in enumerate(idxs):
        ax = axes[i]
        im = ax.imshow(ey_stack[idx].T, origin='lower', cmap='RdBu_r',
                       extent=ext_xy, aspect='auto',
                       vmin=-vmax, vmax=vmax)
        ax.contour(ds['eps_xy'].T, levels=[eps_Si * 0.5],
                   colors='k', linewidths=0.3, alpha=0.5,
                   extent=ext_xy)
        ax.axvline(ds['mon1_x'] * a/1e3, color='cyan', ls='--', lw=0.8)
        ax.axvline(ds['mon2_x'] * a/1e3, color='lime', ls='--', lw=0.8)
        ax.set_ylabel(f"t = {ds['snap_times'][idx]:.0f}", fontsize=9)
        if i == 0:
            ax.set_title(f"{ds['label']}  —  Ey(z=0) snapshots", fontsize=12)
        if i < N_FRAMES_SHOW - 1:
            ax.set_xticks([])
        else:
            ax.set_xlabel('x (µm)')
        plt.colorbar(im, ax=ax, fraction=0.01, pad=0.01)

    plt.savefig(f"{ds['label']}_ey_snapshots.png", dpi=150, bbox_inches='tight')
    plt.show()

## 3. Group Index — Method 1: Broadband Peak Delay

Compute the Hilbert-transform envelope of $E_y(t)$ at each monitor.  
Peak times $t_1$, $t_2$ → group delay $\Delta t = t_2 - t_1$ → $n_g = \Delta t / L_{\text{mon}}$.

This gives a single **broadband** $n_g$ averaged over the pulse bandwidth.

In [ ]:
results_m1 = {}   # store for later comparison

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)

for col, ds in enumerate(datasets):
    t   = ds['t_record']
    ein = ds['ey_t_in']
    eout = ds['ey_t_out']
    L   = ds['L_mon']
    a   = ds['a_nm']

    env_in  = np.abs(hilbert(ein))
    env_out = np.abs(hilbert(eout))

    t1 = t[np.argmax(env_in)]
    t2 = t[np.argmax(env_out)]
    dt_peak = t2 - t1
    ng_brd = dt_peak / L

    results_m1[ds['label']] = dict(ng=ng_brd, t1=t1, t2=t2,
                                    env_in=env_in, env_out=env_out)

    ax = axes[col]
    ax.plot(t, ein,  color='steelblue', lw=0.5, alpha=0.4, label='Mon1 raw')
    ax.plot(t, eout, color='tomato',    lw=0.5, alpha=0.4, label='Mon2 raw')
    ax.plot(t, env_in,  color='steelblue', lw=1.8, label='Mon1 envelope')
    ax.plot(t, env_out, color='tomato',    lw=1.8, label='Mon2 envelope')
    ax.axvline(t1, color='steelblue', ls='--', lw=1)
    ax.axvline(t2, color='tomato',    ls='--', lw=1)
    ymax = max(env_in.max(), env_out.max())
    ax.annotate('', xy=(t2, ymax*0.85), xytext=(t1, ymax*0.85),
                arrowprops=dict(arrowstyle='<->', color='green', lw=1.5))
    ax.text((t1+t2)/2, ymax*0.88, f'Δt={dt_peak:.1f}', ha='center',
            color='green', fontsize=10)
    ax.set_xlabel('Time (Meep units = a/c)')
    ax.set_ylabel('Ey')
    ax.set_title(
        f'{ds["label"]}\na={a:.1f} nm\n'
        f'Δt={dt_peak:.2f}  L={L:.0f}a  →  ng = {ng_brd:.2f}',
        fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Method 1: Broadband ng from Envelope Peak Delay', fontsize=13)
plt.tight_layout()
plt.savefig('fdtd_ng_method1.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Method 1 Summary ===')
for label, r in results_m1.items():
    print(f'  {label}: ng (broadband) = {r["ng"]:.3f}')

## 4. Group Index — Method 2: Bandpass Filter Peak Delay

For each target wavelength $\lambda_c$, apply a narrow Gaussian bandpass (FWHM = 5 nm) in the frequency domain.  
Take the Hilbert envelope of the filtered signal at each monitor, find peak times $t_1(\lambda)$, $t_2(\lambda)$, and compute:

$$n_g(\lambda) = \frac{t_2(\lambda) - t_1(\lambda)}{L_{\text{mon}}}$$

In [ ]:
results_m2 = {}

BW_NM    = 5.0    # Gaussian filter FWHM (nm)
N_WL     = 60     # wavelength points to sweep
WL_PAD   = 8.0    # extend sweep beyond pulse window (nm)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for col, ds in enumerate(datasets):
    t   = ds['t_record']
    ein = ds['ey_t_in']
    eout = ds['ey_t_out']
    L   = ds['L_mon']
    a   = ds['a_nm']
    wl_lo = ds['LAMBDA_MIN_NM'] - WL_PAD
    wl_hi = ds['LAMBDA_MAX_NM'] + WL_PAD

    dt_samp = t[1] - t[0]
    N  = len(t)
    ff = np.fft.rfftfreq(N, d=dt_samp)   # c/a
    Fin  = np.fft.rfft(ein)
    Fout = np.fft.rfft(eout)

    wl_tgt = np.linspace(wl_lo, wl_hi, N_WL)
    ng_m2  = np.full(N_WL, np.nan)

    for i, wl_c in enumerate(wl_tgt):
        f_c   = a / wl_c
        # Convert nm BW to freq-domain sigma
        sig_f = a / wl_c**2 * BW_NM / (2.0 * np.sqrt(2.0 * np.log(2.0)))
        gauss = np.exp(-0.5 * ((ff - f_c) / sig_f) ** 2)

        s_in  = np.fft.irfft(Fin  * gauss, n=N)
        s_out = np.fft.irfft(Fout * gauss, n=N)

        e1 = np.abs(hilbert(s_in))
        e2 = np.abs(hilbert(s_out))

        # Only compute if signal is significant
        if e1.max() < 1e-12 * np.abs(ein).max():
            continue

        t1i = t[np.argmax(e1)]
        t2i = t[np.argmax(e2)]
        ng_m2[i] = (t2i - t1i) / L

    results_m2[ds['label']] = dict(wl=wl_tgt, ng=ng_m2,
                                    ng_brd=results_m1[ds['label']]['ng'])

    ax = axes[col]
    valid = np.isfinite(ng_m2) & (ng_m2 > 1.0) & (ng_m2 < 30.0)
    ax.plot(wl_tgt[valid], ng_m2[valid], 'o-', color='tomato',
            ms=4, lw=1.8, label='FDTD bandpass')
    ax.axhline(results_m1[ds['label']]['ng'], color='green',
               ls='--', lw=1.5, label=f'Method 1 ng={results_m1[ds["label"]]["ng"]:.2f}')
    ax.axvspan(ds['LAMBDA_MIN_NM'], ds['LAMBDA_MAX_NM'],
               alpha=0.12, color='steelblue', label='Pulse window')
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel('Group index $n_g$')
    ax.set_title(f'{ds["label"]}\na={ds["a_nm"]:.1f} nm', fontsize=11)
    ax.set_ylim(0, 15)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Method 2: ng(λ) from Bandpass Filter Peak Delay', fontsize=13)
plt.tight_layout()
plt.savefig('fdtd_ng_method2.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Method 2 Summary ===')
for label, r in results_m2.items():
    valid = np.isfinite(r['ng']) & (r['ng'] > 1) & (r['ng'] < 30)
    if valid.any():
        print(f'  {label}: ng = {np.mean(r["ng"][valid]):.2f} ± {np.std(r["ng"][valid]):.2f}')

## 5. Group Index — Method 3: Spectral Phase Derivative

Transfer function in frequency domain:
$$H(\omega) = \frac{\tilde{E}_y^{\text{out}}(\omega)}{\tilde{E}_y^{\text{in}}(\omega)}$$

Unwrap phase $\phi(\omega) = \arg H(\omega)$, then:
$$n_g(\lambda) = -\frac{1}{L_{\text{mon}}} \frac{d\phi}{d\omega}$$

Phase is unwrapped over $[-\pi, \pi]$ and numerically differentiated. Only frequency bins with sufficient input signal are used.

In [ ]:
results_m3 = {}

SNR_THRESH = 0.05   # min |Fin| relative to peak to include bin

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, ds in enumerate(datasets):
    t   = ds['t_record']
    ein = ds['ey_t_in']
    eout = ds['ey_t_out']
    L   = ds['L_mon']
    a   = ds['a_nm']
    wl_lo = ds['LAMBDA_MIN_NM'] - 10
    wl_hi = ds['LAMBDA_MAX_NM'] + 10

    dt_samp = t[1] - t[0]
    N  = len(t)
    ff = np.fft.rfftfreq(N, d=dt_samp)   # c/a
    Fin  = np.fft.rfft(ein)
    Fout = np.fft.rfft(eout)

    # Transfer function with SNR guard
    snr_mask = np.abs(Fin) > SNR_THRESH * np.abs(Fin).max()
    eps_guard = SNR_THRESH * np.abs(Fin).max()
    denom = np.where(snr_mask, Fin, eps_guard * np.exp(1j * np.angle(Fin)))
    H = Fout / denom

    # Unwrap phase only over the well-sampled region
    phase_raw = np.angle(H)
    phase = np.unwrap(phase_raw)

    # Angular frequency and derivative
    omega = 2.0 * np.pi * ff
    d_phi   = np.gradient(phase, omega)
    ng_ph   = -d_phi / L

    # Convert to wavelength
    wl_all = np.where(ff > 0, a / ff, np.inf)
    mask = (wl_all >= wl_lo) & (wl_all <= wl_hi) & snr_mask

    wl_sel  = wl_all[mask]
    ng_sel  = ng_ph[mask]
    ph_sel  = phase[mask]

    sort_i  = np.argsort(wl_sel)
    wl_sel  = wl_sel[sort_i]
    ng_sel  = ng_sel[sort_i]
    ph_sel  = ph_sel[sort_i]

    results_m3[ds['label']] = dict(wl=wl_sel, ng=ng_sel,
                                    ng_brd=results_m1[ds['label']]['ng'])

    # --- Phase plot ---
    ax = axes[0, col]
    ax.plot(wl_sel, ph_sel, 'steelblue', lw=1.5)
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel('Phase φ(ω) (rad)')
    ax.set_title(f'{ds["label"]}\nUnwrapped phase of H(ω)', fontsize=11)
    ax.grid(True, alpha=0.3)

    # --- ng plot ---
    ax = axes[1, col]
    valid = np.isfinite(ng_sel) & (ng_sel > 1) & (ng_sel < 30)
    ax.plot(wl_sel[valid], ng_sel[valid], color='tomato',
            lw=1.8, label='Phase derivative')
    ax.axhline(results_m1[ds['label']]['ng'], color='green',
               ls='--', lw=1.5, label=f'Method 1 ng={results_m1[ds["label"]]["ng"]:.2f}')
    ax.axvspan(ds['LAMBDA_MIN_NM'], ds['LAMBDA_MAX_NM'],
               alpha=0.12, color='steelblue', label='Pulse window')
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel('Group index $n_g$')
    ax.set_title(f'ng(λ) from phase derivative', fontsize=11)
    ax.set_ylim(0, 15)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Method 3: ng(λ) from Spectral Phase Derivative', fontsize=13)
plt.tight_layout()
plt.savefig('fdtd_ng_method3.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Method 3 Summary ===')
for label, r in results_m3.items():
    valid = np.isfinite(r['ng']) & (r['ng'] > 1) & (r['ng'] < 30)
    if valid.any():
        print(f'  {label}: ng = {np.mean(r["ng"][valid]):.2f} ± {np.std(r["ng"][valid]):.2f}')

## 6. Summary — All Methods Compared

Overlay the three $n_g$ estimates for each dataset to assess consistency.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for col, ds in enumerate(datasets):
    label = ds['label']
    ax = axes[col]

    # Method 1 — horizontal line
    ng1 = results_m1[label]['ng']
    ax.axhline(ng1, color='green', ls='-', lw=2.0,
               label=f'Method 1 (broadband) ng={ng1:.2f}')

    # Method 2 — bandpass
    r2 = results_m2[label]
    v2 = np.isfinite(r2['ng']) & (r2['ng'] > 1) & (r2['ng'] < 30)
    if v2.any():
        ax.plot(r2['wl'][v2], r2['ng'][v2], 'o-', color='steelblue',
                ms=4, lw=1.8, label='Method 2 (bandpass)')

    # Method 3 — phase derivative
    r3 = results_m3[label]
    v3 = np.isfinite(r3['ng']) & (r3['ng'] > 1) & (r3['ng'] < 30)
    if v3.any():
        ax.plot(r3['wl'][v3], r3['ng'][v3], '-', color='tomato',
                lw=1.8, label='Method 3 (phase deriv.)')

    # Pulse window
    ax.axvspan(ds['LAMBDA_MIN_NM'], ds['LAMBDA_MAX_NM'],
               alpha=0.12, color='gold', label='Pulse window')

    ax.set_xlabel('Wavelength (nm)', fontsize=12)
    ax.set_ylabel('Group index $n_g$', fontsize=12)
    ax.set_title(f'{label}\na = {ds["a_nm"]:.1f} nm', fontsize=12)
    ax.set_ylim(0, 15)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Group Index Comparison — All Methods', fontsize=14)
plt.tight_layout()
plt.savefig('fdtd_ng_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Print full table ──────────────────────────────────────────────────
print(f"{'Dataset':<35} {'M1 (broadband)':>16} {'M2 (bandpass)':>16} {'M3 (phase)':>14}")
print('-' * 85)
for ds in datasets:
    lbl = ds['label']
    ng1 = results_m1[lbl]['ng']
    r2  = results_m2[lbl]
    v2  = np.isfinite(r2['ng']) & (r2['ng'] > 1) & (r2['ng'] < 30)
    ng2 = f"{np.mean(r2['ng'][v2]):.2f}±{np.std(r2['ng'][v2]):.2f}" if v2.any() else 'N/A'
    r3  = results_m3[lbl]
    v3  = np.isfinite(r3['ng']) & (r3['ng'] > 1) & (r3['ng'] < 30)
    ng3 = f"{np.mean(r3['ng'][v3]):.2f}±{np.std(r3['ng'][v3]):.2f}" if v3.any() else 'N/A'
    print(f"{lbl:<35} {ng1:>16.3f} {ng2:>16} {ng3:>14}")

## 7. Interactive Animation (optional)

Render the full $E_y$ movie as a JS widget for inline playback.  
Select which dataset to animate by setting `DATASET_IDX`.

In [ ]:
DATASET_IDX = 0    # 0 = top1, 1 = top2

ds = datasets[DATASET_IDX]
a  = ds['a_nm']
rx = ds['snap_region_x'] * a / 1e3
ry = ds['snap_region_y'] * a / 1e3
ext_xy = [-rx/2, rx/2, -ry/2, ry/2]

ey_stack = ds['ey_snapshots']
snaptimes = ds['snap_times']
vmax = np.percentile(np.abs(ey_stack), 99.5)

fig, ax = plt.subplots(figsize=(16, 3))
im = ax.imshow(ey_stack[0].T, origin='lower', cmap='RdBu_r',
               extent=ext_xy, aspect='auto', vmin=-vmax, vmax=vmax)
ax.contour(ds['eps_xy'].T, levels=[eps_Si * 0.5], colors='k',
           linewidths=0.3, alpha=0.4, extent=ext_xy)
ax.axvline(ds['mon1_x'] * a/1e3, color='cyan', ls='--', lw=1, alpha=0.7)
ax.axvline(ds['mon2_x'] * a/1e3, color='lime', ls='--', lw=1, alpha=0.7)
ax.set_xlabel('x (µm)')
ax.set_ylabel('y (µm)')
ttl = ax.set_title(f'Ey(z=0)   {ds["label"]}   t={snaptimes[0]:.0f}')
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)

def _update(frame):
    im.set_data(ey_stack[frame].T)
    ttl.set_text(f'Ey(z=0)   {ds["label"]}   t={snaptimes[frame]:.0f}')
    return [im, ttl]

ani = animation.FuncAnimation(fig, _update, frames=len(ey_stack),
                               interval=80, blit=True)
plt.tight_layout()
plt.close()   # suppress static display
HTML(ani.to_jshtml())